In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c ieee-fraud-detection
!unzip ieee-fraud-detection.zip -d ./data


100% 118M/118M [00:00<00:00, 148MB/s]

Archive:  ieee-fraud-detection.zip
  inflating: ./data/sample_submission.csv  
  inflating: ./data/test_identity.csv  
  inflating: ./data/test_transaction.csv  
  inflating: ./data/train_identity.csv  
  inflating: ./data/train_transaction.csv  


In [6]:
import argparse
import json
import os
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REVIEW_COST = 5.00
FRICTION_COST = 3.00
FP_COST = REVIEW_COST + FRICTION_COST

ENTITY_COLS = ["card1", "addr1", "P_emaildomain"]
WINDOWS = [3600, 86400, 7 * 86400]


def load_data(data_dir, sample_frac=1.0):
    tx_path = os.path.join(data_dir, "train_transaction.csv")
    if not os.path.exists(tx_path):
        raise FileNotFoundError(
            f"Could not find {tx_path}.\n"
            "Download IEEE-CIS Fraud Detection from Kaggle and unzip into --data-dir."
        )
    print(f"[load] reading {tx_path} ...")
    df = pd.read_csv(tx_path)

    id_path = os.path.join(data_dir, "train_identity.csv")
    if os.path.exists(id_path):
        print(f"[load] merging {id_path} ...")
        idf = pd.read_csv(id_path)
        df = df.merge(idf, on="TransactionID", how="left")
    else:
        print("[load] train_identity.csv not found -- continuing without it (fine).")

    if sample_frac < 1.0:
        cutoff = df["TransactionDT"].quantile(sample_frac)
        df = df[df["TransactionDT"] <= cutoff].copy()
        print(f"[load] sampled earliest {sample_frac:.0%} of the time range")

    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = df[c].astype(np.float32)

    df = df.sort_values("TransactionDT").reset_index(drop=True)
    print(f"[load] {len(df):,} rows | fraud rate {df['isFraud'].mean():.4%}")
    return df


def _prior_window_counts(df, col, window):
    out = np.full(len(df), np.nan, dtype=np.float32)
    t_all = df["TransactionDT"].values
    for _, idx in df.groupby(col, sort=False).indices.items():
        t = t_all[idx]
        left = np.searchsorted(t, t - window, side="left")
        out[idx] = np.arange(len(t)) - left
    return out


def add_velocity_features(df):
    print("[feat] building backward-looking entity-velocity features ...")
    for col in ENTITY_COLS:
        if col not in df.columns:
            continue
        g = df.groupby(col, sort=False)

        df[f"{col}__n_prior"] = g.cumcount().astype(np.float32)

        df[f"{col}__secs_since_prev"] = (
            df["TransactionDT"] - g["TransactionDT"].shift(1)
        ).astype(np.float32)

        csum = g["TransactionAmt"].cumsum() - df["TransactionAmt"]
        cnt = df[f"{col}__n_prior"].replace(0, np.nan)
        df[f"{col}__prior_amt_mean"] = (csum / cnt).astype(np.float32)

        df[f"{col}__amt_vs_prior"] = (
            df["TransactionAmt"] / df[f"{col}__prior_amt_mean"]
        ).astype(np.float32)

        for w in WINDOWS:
            df[f"{col}__cnt_{w}s"] = _prior_window_counts(df, col, w)

    df["hour"] = ((df["TransactionDT"] / 3600) % 24).astype(np.int8)
    df["dow"] = ((df["TransactionDT"] / 86400) % 7).astype(np.int8)
    df["amt_log"] = np.log1p(df["TransactionAmt"]).astype(np.float32)
    df["amt_decimal"] = (
        df["TransactionAmt"] - df["TransactionAmt"].astype(int)
    ).astype(np.float32)

    return df


def assert_no_future_leakage(df, n_checks=300, seed=0):
    print("[test] verifying features are strictly backward-looking ...")
    rng = np.random.default_rng(seed)
    col, w = ENTITY_COLS[0], WINDOWS[1]
    feat = f"{col}__cnt_{w}s"
    if feat not in df.columns:
        print("[test] skipped (feature absent)")
        return

    sub = df[[col, "TransactionDT", feat]].dropna(subset=[col])
    rows = rng.choice(len(sub), size=min(n_checks, len(sub)), replace=False)
    vals = sub[col].values
    times = sub["TransactionDT"].values
    got = sub[feat].values

    for i in rows:
        mask = (vals == vals[i]) & (times < times[i]) & (times >= times[i] - w)
        expected = int(mask.sum())
        assert got[i] == expected, (
            f"LEAKAGE at row {i}: feature={got[i]} but brute-force past-only={expected}"
        )
    print(f"[test] PASSED on {len(rows)} sampled rows -- no future information used.")


def prepare_matrix(df):
    drop = {"TransactionID", "isFraud"}
    feats = [c for c in df.columns if c not in drop]

    X = df[feats].copy()
    for c in X.select_dtypes(include=["object"]).columns:
        X[c] = X[c].astype("category")
    return X, df["isFraud"].values.astype(int), feats


def temporal_split(df, train_end=0.70, calib_end=0.85):
    t = df["TransactionDT"]
    q1, q2 = t.quantile(train_end), t.quantile(calib_end)
    tr = (t <= q1).values
    ca = ((t > q1) & (t <= q2)).values
    te = (t > q2).values
    print(
        f"[split] train {tr.sum():,} (fraud {df.loc[tr,'isFraud'].mean():.3%}) | "
        f"calib {ca.sum():,} (fraud {df.loc[ca,'isFraud'].mean():.3%}) | "
        f"test {te.sum():,} (fraud {df.loc[te,'isFraud'].mean():.3%})"
    )
    return tr, ca, te


def train_model(X_tr, y_tr, X_ca, y_ca, amt_tr, cost_weighted=True):
    import lightgbm as lgb

    pos, neg = y_tr.sum(), len(y_tr) - y_tr.sum()
    spw = neg / max(pos, 1)

    if cost_weighted:
        w = np.ones(len(y_tr), dtype=np.float32)
        w[y_tr == 1] = np.clip(amt_tr[y_tr == 1] / FP_COST, 1.0, 200.0)
        w[y_tr == 0] = 1.0
    else:
        w = None

    params = dict(
        objective="binary",
        metric="average_precision",
        learning_rate=0.05,
        num_leaves=128,
        min_child_samples=100,
        feature_fraction=0.7,
        bagging_fraction=0.8,
        bagging_freq=1,
        lambda_l2=1.0,
        scale_pos_weight=spw if not cost_weighted else 1.0,
        n_jobs=-1,
        verbose=-1,
        seed=42,
    )

    dtr = lgb.Dataset(X_tr, y_tr, weight=w)
    dca = lgb.Dataset(X_ca, y_ca, reference=dtr)

    print(f"[train] LightGBM | pos_weight={spw:.1f} | cost_weighted={cost_weighted}")
    model = lgb.train(
        params, dtr, num_boost_round=3000, valid_sets=[dca],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)],
    )
    print(f"[train] best iteration: {model.best_iteration}")
    return model


def calibrate(model, X_ca, y_ca):
    from sklearn.isotonic import IsotonicRegression

    raw = model.predict(X_ca, num_iteration=model.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    iso.fit(raw, y_ca)
    return iso


def calibration_report(y, p, n_bins=15):
    from sklearn.metrics import brier_score_loss

    bins = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.digitize(p, bins[1:-1]), 0, len(bins) - 2)
    rows = []
    for b in range(len(bins) - 1):
        m = idx == b
        if m.sum() == 0:
            continue
        rows.append({"mean_pred": float(p[m].mean()),
                     "observed": float(y[m].mean()),
                     "n": int(m.sum())})
    ece = sum(r["n"] * abs(r["mean_pred"] - r["observed"]) for r in rows) / len(y)
    return {"brier": float(brier_score_loss(y, p)), "ece": float(ece), "bins": rows}


def expected_cost(y, p, amount, threshold):
    flag = p >= threshold
    fp = flag & (y == 0)
    fn = (~flag) & (y == 1)
    return float(fp.sum() * FP_COST + amount[fn].sum())


def sweep_threshold(y, p, amount, n=400):
    grid = np.unique(np.quantile(p, np.linspace(0.50, 0.9999, n)))
    costs = np.array([expected_cost(y, p, amount, t) for t in grid])
    best = int(np.argmin(costs))
    return grid, costs, float(grid[best]), float(costs[best])


def f1_optimal_threshold(y, p):
    from sklearn.metrics import precision_recall_curve

    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    return float(thr[int(np.argmax(f1[:-1]))])


def flag_everything_cost(y):
    return float((y == 0).sum() * FP_COST)


def capacity_analysis(y, p, amount, capacity_fracs=(0.001, 0.005, 0.01, 0.02, 0.05)):
    total_fraud_dollars = amount[y == 1].sum()
    out = []
    for frac in capacity_fracs:
        k = max(int(len(y) * frac), 1)

        by_score = np.argsort(-p)[:k]
        by_ev = np.argsort(-(p * amount))[:k]
        rng = np.random.default_rng(42)
        by_rand = rng.choice(len(y), size=k, replace=False)

        out.append({
            "capacity_frac": frac,
            "k": k,
            "random_dollars_recovered": float(amount[by_rand][y[by_rand] == 1].sum()),
            "score_precision_at_k": float(y[by_score].mean()),
            "score_dollars_recovered": float(amount[by_score][y[by_score] == 1].sum()),
            "score_recall_dollars": float(
                amount[by_score][y[by_score] == 1].sum() / total_fraud_dollars),
            "ev_precision_at_k": float(y[by_ev].mean()),
            "ev_dollars_recovered": float(amount[by_ev][y[by_ev] == 1].sum()),
            "ev_recall_dollars": float(
                amount[by_ev][y[by_ev] == 1].sum() / total_fraud_dollars),
        })
    return out, float(total_fraud_dollars)


def latency_benchmark(model, X_te, n=2000):
    print("[latency] measuring single-transaction inference ...")
    sample = X_te.iloc[:n]
    times = []
    for i in range(len(sample)):
        row = sample.iloc[[i]]
        t0 = time.perf_counter()
        model.predict(row, num_iteration=model.best_iteration)
        times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    return {"p50_ms": float(np.percentile(times, 50)),
            "p95_ms": float(np.percentile(times, 95)),
            "p99_ms": float(np.percentile(times, 99))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="./data")
    ap.add_argument("--sample-frac", type=float, default=1.0)
    ap.add_argument("--out", default="./results")
    ap.add_argument("--skip-uncalibrated-ablation", action="store_true")

    args = ap.parse_args(['--data-dir', './data', '--sample-frac', '0.25'])

    os.makedirs(args.out, exist_ok=True)

    df = load_data(args.data_dir, args.sample_frac)
    df = add_velocity_features(df)
    assert_no_future_leakage(df)

    X, y, feats = prepare_matrix(df)
    amount = df["TransactionAmt"].values.astype(np.float64)
    tr, ca, te = temporal_split(df)

    ca_pos = np.where(ca)[0]
    cut = int(len(ca_pos) * 0.6)
    ca_fit = np.zeros(len(df), dtype=bool); ca_fit[ca_pos[:cut]] = True
    ca_sel = np.zeros(len(df), dtype=bool); ca_sel[ca_pos[cut:]] = True
    print(f"[split] calib -> isotonic-fit {ca_fit.sum():,} | "
          f"threshold-select {ca_sel.sum():,}")

    model = train_model(X[tr], y[tr], X[ca_fit], y[ca_fit], amount[tr], cost_weighted=True)

    raw_te = model.predict(X[te], num_iteration=model.best_iteration)
    iso = calibrate(model, X[ca_fit], y[ca_fit])
    cal_te = iso.predict(raw_te)

    raw_sel = model.predict(X[ca_sel], num_iteration=model.best_iteration)
    cal_sel = iso.predict(raw_sel)

    from sklearn.metrics import average_precision_score, roc_auc_score
    results = {
        "n_train": int(tr.sum()), "n_calib": int(ca.sum()), "n_test": int(te.sum()),
        "test_fraud_rate": float(y[te].mean()),
        "pr_auc": float(average_precision_score(y[te], raw_te)),
        "roc_auc": float(roc_auc_score(y[te], raw_te)),
        "cost_assumptions": {"review_cost": REVIEW_COST,
                             "friction_cost": FRICTION_COST,
                             "fn_cost": "full transaction amount"},
    }
    print(f"\n[metrics] PR-AUC {results['pr_auc']:.4f} | ROC-AUC {results['roc_auc']:.4f}")

    results["calibration_raw"] = calibration_report(y[te], raw_te)
    results["calibration_isotonic"] = calibration_report(y[te], cal_te)
    print(f"[metrics] Brier raw {results['calibration_raw']['brier']:.5f} "
          f"-> isotonic {results['calibration_isotonic']['brier']:.5f}")
    print(f"[metrics] ECE   raw {results['calibration_raw']['ece']:.5f} "
          f"-> isotonic {results['calibration_isotonic']['ece']:.5f}")

    t_star = sweep_threshold(y[ca_sel], cal_sel, amount[ca_sel])[2]
    c_deployed = expected_cost(y[te], cal_te, amount[te], t_star)

    c_flagall = flag_everything_cost(y[te])
    c_donothing = float(amount[te][y[te] == 1].sum())
    c_half = expected_cost(y[te], cal_te, amount[te], 0.5)
    t_f1 = f1_optimal_threshold(y[ca_sel], cal_sel)
    c_f1 = expected_cost(y[te], cal_te, amount[te], t_f1)

    grid, costs, t_oracle, c_oracle = sweep_threshold(y[te], cal_te, amount[te])

    results["threshold"] = {
        "t_selected_on_calib": t_star,
        "cost_deployed": c_deployed,
        "t_f1_baseline": t_f1,
        "cost_f1_baseline": c_f1,
        "cost_at_0.5": c_half,
        "cost_do_nothing": c_donothing,
        "cost_flag_everything": c_flagall,
        "t_oracle_on_test": t_oracle,
        "cost_oracle_on_test": c_oracle,
        "savings_vs_f1_pct": float(100 * (c_f1 - c_deployed) / c_f1),
        "oracle_gap_pct": float(100 * (c_deployed - c_oracle) / max(c_oracle, 1e-9)),
    }
    print(f"\n[cost] flag everything          ${c_flagall:,.0f}")
    print(f"[cost] do nothing               ${c_donothing:,.0f}")
    print(f"[cost] fixed threshold 0.5      ${c_half:,.0f}  (degenerate at this base rate)")
    print(f"[cost] F1-optimal  t={t_f1:.4f}   ${c_f1:,.0f}  <- the honest baseline")
    print(f"[cost] cost-optimal t={t_star:.4f}  ${c_deployed:,.0f}  "
          f"({results['threshold']['savings_vs_f1_pct']:.1f}% better than F1-optimal)")
    print(f"[cost] test-set ORACLE t={t_oracle:.4f} ${c_oracle:,.0f}  "
          f"(gap {results['threshold']['oracle_gap_pct']:.1f}% -- disclose this)")

    if not args.skip_uncalibrated_ablation:
        t_raw = sweep_threshold(y[ca_sel], raw_sel, amount[ca_sel])[2]
        cost_no_cal = expected_cost(y[te], raw_te, amount[te], t_raw)
        results["calibration_ablation"] = {
            "threshold_selected_on_raw_scores": t_raw,
            "threshold_selected_on_calibrated": t_star,
            "cost_without_calibration": cost_no_cal,
            "cost_with_calibration": c_deployed,
            "calibration_saves_pct": float(
                100 * (cost_no_cal - c_deployed) / max(cost_no_cal, 1e-9)),
        }
        print(f"[cost] same pipeline WITHOUT calibration: ${cost_no_cal:,.0f} "
              f"({results['calibration_ablation']['calibration_saves_pct']:.1f}% worse)")

    cap, total_fraud = capacity_analysis(y[te], cal_te, amount[te])
    results["capacity"] = cap
    results["total_fraud_dollars_test"] = total_fraud
    print(f"\n[capacity] total fraud in test window: ${total_fraud:,.0f}")
    print(f"{'cap':>6} {'k':>7} {'P@k(score)':>11} {'$rec(score)':>13} "
          f"{'P@k(EV)':>9} {'$rec(EV)':>13} {'$rec(rand)':>12}")
    for r in cap:
        print(f"{r['capacity_frac']:>6.3f} {r['k']:>7,} "
              f"{r['score_precision_at_k']:>11.3f} {r['score_dollars_recovered']:>13,.0f} "
              f"{r['ev_precision_at_k']:>9.3f} {r['ev_dollars_recovered']:>13,.0f} "
              f"{r['random_dollars_recovered']:>12,.0f}")

    results["latency"] = latency_benchmark(model, X[te])
    print(f"\n[latency] p50 {results['latency']['p50_ms']:.2f}ms | "
          f"p95 {results['latency']['p95_ms']:.2f}ms | "
          f"p99 {results['latency']['p99_ms']:.2f}ms")

    imp = pd.DataFrame({
        "feature": model.feature_name(),
        "gain": model.feature_importance("gain"),
    }).sort_values("gain", ascending=False).head(30)
    imp.to_csv(os.path.join(args.out, "feature_importance.csv"), index=False)
    results["top_features"] = imp["feature"].head(15).tolist()
    print("\n[features] top 15 by gain:")
    for f in results["top_features"]:
        print(f"   - {f}")

    with open(os.path.join(args.out, "results.json"), "w") as fh:
        json.dump(results, fh, indent=2)
    model.save_model(os.path.join(args.out, "model.txt"))

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(1, 4, figsize=(21, 4.5))

        for key, label in [("calibration_raw", "raw"), ("calibration_isotonic", "isotonic")]:
            b = results[key]["bins"]
            ax[0].plot([r["mean_pred"] for r in b], [r["observed"] for r in b],
                       marker="o", ms=3, label=label)
        ax[0].plot([1e-4, 1], [1e-4, 1], "k--", lw=1)
        ax[0].set_xscale("log"); ax[0].set_yscale("log")
        ax[0].set_xlabel("predicted P(fraud)"); ax[0].set_ylabel("observed rate")
        ax[0].set_title("Reliability diagram"); ax[0].legend()

        ax[1].plot(grid, costs, label="cost on test")
        ax[1].axvline(t_star, color="r", ls="--",
                      label=f"t chosen on calib = {t_star:.3f}")
        ax[1].axvline(t_oracle, color="g", ls="-.", lw=1,
                      label=f"test oracle = {t_oracle:.3f}")
        ax[1].axhline(c_f1, color="gray", ls=":", label="F1-optimal baseline")
        ax[1].set_xlabel("threshold"); ax[1].set_ylabel("expected cost ($)")
        ax[1].set_title("Expected cost vs threshold"); ax[1].legend(fontsize=8)

        fr = [r["capacity_frac"] for r in cap]
        ax[2].plot(fr, [r["score_recall_dollars"] for r in cap], marker="o",
                   label="rank by score")
        ax[2].plot(fr, [r["ev_recall_dollars"] for r in cap], marker="s",
                   label="rank by score x amount")
        ax[2].set_xscale("log")
        ax[2].set_xlabel("review capacity (fraction of txns)")
        ax[2].set_ylabel("fraud dollars recovered (share)")
        ax[2].plot(fr, [r["random_dollars_recovered"] / total_fraud for r in cap],
                   marker="^", ls=":", color="gray", label="random (floor)")
        ax[2].set_title("Capacity-constrained triage"); ax[2].legend(fontsize=8)

        from sklearn.metrics import precision_recall_curve
        pr, rc, _ = precision_recall_curve(y[te], cal_te)
        ax[3].plot(rc, pr, label=f"PR-AUC = {results['pr_auc']:.3f}")
        ax[3].axhline(y[te].mean(), color="k", ls="--", lw=1,
                      label=f"base rate = {y[te].mean():.3f}")
        ax[3].set_xlabel("recall"); ax[3].set_ylabel("precision")
        ax[3].set_title("Precision-Recall (test)"); ax[3].legend(fontsize=8)

        plt.tight_layout()
        plt.savefig(os.path.join(args.out, "figures.png"), dpi=140)
        print(f"\n[out] figures -> {args.out}/figures.png")
    except Exception as e:
        print(f"[warn] plotting skipped: {e}")

    print(f"[out] results -> {args.out}/results.json")


if __name__ == "__main__":
    main()

[load] reading ./data/train_transaction.csv ...
[load] merging ./data/train_identity.csv ...
[load] sampled earliest 25% of the time range
[load] 147,635 rows | fraud rate 2.6335%
[feat] building backward-looking entity-velocity features ...
[test] verifying features are strictly backward-looking ...
[test] PASSED on 300 sampled rows -- no future information used.
[split] train 103,345 (fraud 2.530%) | calib 22,144 (fraud 2.082%) | test 22,146 (fraud 3.667%)
[split] calib -> isotonic-fit 13,286 | threshold-select 8,858
[train] LightGBM | pos_weight=38.5 | cost_weighted=True
[200]	valid_0's average_precision: 0.381924
[400]	valid_0's average_precision: 0.423368
[600]	valid_0's average_precision: 0.43497
[800]	valid_0's average_precision: 0.44261
[1000]	valid_0's average_precision: 0.447134
[train] best iteration: 1092

[metrics] PR-AUC 0.6235 | ROC-AUC 0.8967
[metrics] Brier raw 0.02247 -> isotonic 0.02080
[metrics] ECE   raw 0.02175 -> isotonic 0.01074

[cost] flag everything          